<a href="https://colab.research.google.com/github/ss01-0-0/Pilot/blob/main/Python-Projects/Stock%20Market%20Data%20Presenter.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# import libraries that i will use
import yfinance as yf
import pandas as pd
import matplotlib.pyplot as plt
import logging as lg

# get user to input ticker symbol, start and end date
ticker_symbol=input("Ticker symbol to check: ")
start_date=input("Start date (YYYY-MM-DD): ")
end_date=input("End date (YYYY-MM-DD): ")

lg.getLogger("yfinance").setLevel(lg.CRITICAL)

# fetch data
data=yf.download(ticker_symbol,start=start_date,end=end_date)
if data.empty:
  print("""]
Error, data typed is in the wrong format or hasn't been found.
Please try again.""")
else:
  # compute analysis columns, and moving average signals (for using .describe()
  # later)
  data["Return"]=data["Close"].pct_change()
  data["MA20"]=data["Close"].rolling(20).mean()
  data["MA5"]=data["Close"].rolling(5).mean()
  data["Volatility"]=data["Return"].rolling(20).std()
  data["Signal"]=0
  data.loc[data["MA5"]>data["MA20"], "Signal"]=1
  data.loc[data["MA5"]<data["MA20"], "Signal"]=02
  data["Signal Change"]=data["Signal"].diff()
  golden_cross=data[data["Signal Change"]==2]
  death_cross=data[data["Signal Change"]==-2]
  # backtest - strategy return versus buy and hold
  data["Strategy_Return"]= data["Signal"].shift(1) * data["Return"]
  cumulative_strategy=(1+data["Strategy_Return"]).cumprod()-1
  cumulative_holding=(1+data["Return"]).cumprod()-1
  final_strategy_return = cumulative_strategy.iloc[-1]
  final_holding_return = cumulative_holding.iloc[-1]
  print(f"\nStrategy return over period: {final_strategy_return:.2%}")
  print(f"Buy-and-hold return over period: {final_holding_return:.2%}")
  # risk-adjusted metrics
  sharpe_ratio = (data["Strategy_Return"].mean() / data["Strategy_Return"].std()) * (252 ** 0.5)
  equity_curve = 1 + cumulative_strategy
  running_max_equity = equity_curve.cummax()
  drawdown = (equity_curve - running_max_equity) / running_max_equity
  max_drawdown = drawdown.min()
  print(f"Sharpe Ratio: {sharpe_ratio:.2f}")
  print(f"Max Drawdown: {max_drawdown:.2%}")
  # plot the data (for close price, MA20, MA5) as well as golden cross and death
  # cross
  plt.figure(figsize=(10,5))
  plt.plot(data["Close"], label="Close Price", alpha=0.6)
  plt.plot(data["MA20"], label="20-day MA")
  plt.plot(data["MA5"], label="5-day MA")
  plt.scatter(golden_cross.index, golden_cross["Close"], color="green", marker="^", s=100, label="Golden Cross")
  plt.scatter(death_cross.index, death_cross["Close"], color="red", marker="v", s=100, label="Death Cross")
  plt.title(f"{ticker_symbol} Moving Crossover Signal")
  plt.xlabel("Date")
  plt.ylabel("Price")
  plt.legend()
  plt.show()
  # plot the data (for volatility)
  plt.figure(figsize=(10,3))
  plt.plot(data["Volatility"], label="20-day Volatility", color="orange")
  plt.title(f"{ticker_symbol} Rolling Volatility")
  plt.legend()
  plt.show()

# What I Learned
- Python becomes much more powerful through external libraries.
- yfinance provides an easy interface to retrieve historical stock market data.
- A pandas DataFrame stores tabular data.
- .head() returns the first five rows by default, making it useful for inspecting datasets quickly.
- .download() downloads the data from the given DataFrame.
- .describe() presents count, mean, quartiles etc.
- .rolling() when paired with a statistical function such as .mean() or .std() is a way of going through a subset while carrying out the statistical function paired with it.
- .figure is a more customisable way of creating an empty space to plot a graph on.
- .plot is how you can plot points or a line graph.
- learned what MA20, MA5, crossover signals, and close price mean for traders
- learned how to interpret volatility graphs and crossover signals in conjunction to decide on whether buying or selling is better
- learned how to use .loc[] and .scatter() with .diff() to plot crosses
- learned how to use logging as a Python library.
- learned how to use .shift() to move data in a dataframe.
- learned how to use .iloc() to locate integers in a dataframe.
- learned how to interpret results of backtesting to understand what the correct strategy in certain situations is.
- a short position paired with extreme single-day price move can produce a model loss that is impossible.
- sometimes outperforming buy-and-hold and being "safe" aren't the same - (refer to **Findings** section below)
- risk-adjusted return using (Sharpe Ratio) can help someone observe a different aspect of raw return where sometimes the strategy might underperform in absolute terms but can have very good risk-return (refer to **Findings** section)


---


# Version Changes
v.01:
- is able to import yfinance, pandas and matplotlib
- is able to access data from yfinance
- is able to display the first 5 rows from the dataset
- cannot use user input to pull specific data

v.02:
- matplotlib import is corrected to say matplotlib.pyplot
- allows user to enter ticker symbol, dates, and number of days to pull data for.
- is able to describe data.
- cannot present the data in graphs yet.
- columns are not aligned in the terminal (at least on my screen)

v.03:
- added the ability to plot data
- removed .describe() for now
- added variables for volatility, MA20 and returns
- for .std needed to add () so that it's not using the method itself, but is calling it
- code reformatted

v.04:
- added MA5, crossover signals (such as golden cross or death cross)
- added a separate graph showing volatility
- code reformatted
- if else check to test if the data inputted is correct (avoids crashes)
- to make it visually cleaner used logging to clear the screen of the error message should it come up

v.05:
- added backtesting (strategy return versus buy and hold)
- some minor typos corrected

v.06:
- Sharpe Ratio calculation (annualized, using the standard `sqrt(252)`
  trading-day convention) to measure risk-adjusted return, not just raw
  return
- Max Drawdown calculation (corrected version) to measure the worst
  peak-to-trough loss an investor would have experienced holding the
  strategy
- Tested 4 different regime cases and tried to interpret the data (look below)

## Findings

This project backtests a simple 5-day/20-day moving average crossover
strategy against four historically distinct market regimes, to test
whether — and when — the strategy provides real value versus simply
buying and holding.

| Regime | Ticker | Period | Strategy Return | Buy-and-Hold | Sharpe Ratio | Max Drawdown |
|---|---|---|---|---|---|---|
| Flat / sideways | IBM | 2015–2016 | -26.84% | +9.80% | -0.69 | -41.52% |
| Extreme mania | GME | Oct 2020–Jan 2021 | +1380.51% | +3226.51% | 3.59 | -44.29% |
| Steady uptrend | AAPL | 2023 | +8.51% | +54.80% | 0.53 | -25.26% |
| Sustained downtrend | META | 2022 | -28.24% | -64.45% | -0.22 | -96.71% |

### Key insights

- **No single regime favors the strategy outright.** Across four genuinely
  different market conditions, the strategy never consistently beat simple
  buy-and-hold on raw return — but its behavior in each case was
  explainable, not random.
- **Flat markets punish the strategy hardest** (IBM): a negative Sharpe
  Ratio means the strategy didn't even compensate for the risk taken,
  driven by "whipsaw" — the short-term average repeatedly crossing the
  long-term one on meaningless noise rather than a real trend.
- **Return and risk-adjusted return can tell different stories** (GME):
  the strategy earned far less total profit than buy-and-hold during the
  short squeeze, but its Sharpe Ratio (3.59) was the best of all four
  tests — meaning it earned strong, efficient returns relative to the
  risk taken, even though it couldn't front-run the full extremity of the
  move.
- **Beating buy-and-hold isn't the same as being safe** (META): the
  strategy technically outperformed buy-and-hold here, but its -96.71%
  max drawdown reveals it came close to a near-total loss at its worst
  point — a genuinely dangerous result hidden behind a headline "win."
- **Trend-following strategies structurally lag.** In AAPL's steady
  uptrend, the strategy was profitable but captured only a fraction of
  the available gain, reflecting the inherent cost of waiting for a
  confirmed crossover before acting.

### Honest conclusion

This strategy, as implemented, has no consistent edge over simply buying
and holding. Its value is context-dependent: strongest in extreme,
volatile conditions when judged by risk-adjusted return, weakest in flat,
directionless markets, and capable of carrying hidden risk even when it
"wins" on paper. Building a genuinely robust strategy from here would
likely require a regime filter (e.g. using the existing `Volatility`
column to avoid trading in low-trend conditions) rather than relying on
a single indicator alone.